Install required dependencies

In [ ]:
%pip install -r req.txt

  Using cached aiohttp-3.12.15-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached charset_normalizer-3.4.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (36 kB)
  Using cached db_dtypes-1.4.3-py3-none-any.whl.metadata (3.0 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached frozenlist-1.7.0-cp313-cp313-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached google_ai_generativelanguage-0.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.

Import environment variables

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEYS = os.getenv('GEMINI_API_KEYS')
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET = os.getenv('DATASET')
TABLE = os.getenv('TABLE')
REGION = os.getenv('REGION')

Fetch the data

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://services.google.com/fh/files/misc/startup_technical_guide_ai_agents_final.pdf")

documents = loader.load()

/home/busycaesar/projects/personal/embeddings-cosine/TorontoJS/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Split the data into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

Store the data into vector database

In [6]:
from langchain_google_community import BigQueryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

bq_vector_store = BigQueryVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding_model
)

bq_vector_store.add_documents(chunks)

BigQuery table torontojs-488120.torontojs_vectordb.content initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=torontojs-488120&ws=!1m5!1m4!4m3!1storontojs-488120!2storontojs_vectordb!3scontent


['dc800161be544f6b8777d7d85951e404',
 '05d28006bb474f49ba71f225e3c86276',
 '188d361456e146b6aa689ad8b01336cf',
 'd4660fef58074470ad71e6ed53760f8f',
 '83503df594d44a28af8db61d8eb045b6',
 '73783f5f93584c17a960f23443955bbf',
 '2d934841f0da4399ab599969a2944f4c',
 'bab954fb31f841138fcf7a27ade4507b',
 'd5fa70cacbeb4579893dc36f5b61a0fe',
 'be61f90d6c494fb09a4526840559353f',
 'b8da52d0495f49a694218ca678ebcdc6',
 '47cd574fce2c40f8879f0d8e96146a1c',
 'ea3dcc88876343bba224160e95c00ee9',
 'af9e5a8846b245d7b1c166a7fc6375cf',
 '175b307f63f44f87811296e42e13dd00',
 'f303c96cdc954bddacb42e77f2c2d3f8',
 '7d6287d2afeb47a2aaa800001cd23aa2',
 '995929a33b8742a2bead6b74b163873a',
 'ce88dba6574446d0a5742f9b9b1e62c4',
 '27301bcb32c143768b267ed05799114d',
 '0c9dea8fd3ef426f91c0014d1d643ab8',
 '7f0b8eafc82c4695a4ddcff818934f47',
 'a3a4a5ec8f33434bbed8caa843491796',
 '9e8e121bc0104a0c9c737a718b293eaa',
 '7361c93bcbef4bc7b849bc18879cb515',
 '3de075c7784943339eb64e7eb8464372',
 '9bf6a559e9fb419ba407e8da68aaf082',
 

User's query

In [14]:
user_query = "What are the Core components for building AI agents?"

Fetch the relevant chunk of data

In [16]:
retrieved_docs = bq_vector_store.as_retriever().invoke(user_query)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [17]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"], 
    template= 
    """
        Use the following pieces of context to answer the question at the end.

        Context: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import clear_output

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=GEMINI_API_KEYS)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)

The core components for building AI agents are:

*   A core reasoning model
*   A set of tools to enable action
*   Data architecture options for short-term and long-term agent memory
*   A grounding mechanism to ensure factual accuracy
*   Deployment options
